## Taran's Contribution

## Transfer learnt model built on a pretrained LLM such as GPT-2 or TinyLlama using fine-tuning mechanisms such as PEFT (LoRA, QLoRA) etc.

In [1]:
pip install transformers datasets peft accelerate evaluate bert_score rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 100.3 MB/s eta 0:00:00
  Cre

#Loading Data

In [2]:
import pandas as pd
from datasets import Dataset

# Load training passages and test QA data
passages_df = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")
test_df = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet")

train_data_sample = passages_df.sample(frac=0.1, random_state=42)  # frac=0.1 samples 10% of the data

# Convert pandas DataFrames to Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_data_sample[['passage']].rename(columns={'passage': 'text'}))
test_dataset = Dataset.from_pandas(test_df[['question']].rename(columns={'question': 'text'}))

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
print(passages_df.columns)

Index(['passage'], dtype='object')


In [4]:
passages_df.head()

,passage
id,
9797,New data on viruses isolated from patients wit...
11906,We describe an improved method for detecting d...
16083,We have studied the effects of curare on respo...
23188,Kinetic and electrophoretic properties of 230-...
23469,Male Wistar specific-pathogen-free rats aged 2...


In [5]:
print(test_df.columns)

Index(['question', 'answer', 'relevant_passage_ids'], dtype='object')


In [6]:
test_df.iloc[0]

,0
question,Is Hirschsprung disease a mendelian or a multi...
answer,"Coding sequence mutations in RET, GDNF, EDNRB,..."
relevant_passage_ids,"[20598273, 6650562, 15829955, 15617541, 230011..."


# Load GPT-2 model and tokenizer

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM # Import the necessary classes
model_name = "gpt2"  # Use the model of your choice
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

# Apply LoRA to the model

In [8]:

from peft import LoraConfig, get_peft_model

# LoRA fine-tuning applied on the pre-trained GPT-2 model
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model_lora = get_peft_model(model, lora_config)

/usr/local/lib/python3.11/dist-packages/peft/tuners/lora/layer.py:1803: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


# Tokenization function

In [9]:

def tokenize_data(examples):
    tokenized_inputs = tokenizer(examples['text'], truncation=True, padding=True, max_length=512)
    tokenized_inputs["labels"] = tokenized_inputs["input_ids"].copy()
    return tokenized_inputs

# Ensure the tokenizer has a padding token
tokenizer.pad_token = tokenizer.eos_token

# Tokenize the passages and test data
train_data = train_dataset.map(tokenize_data, batched=True)
test_data = test_dataset.map(tokenize_data, batched=True)

Map:   0%|          | 0/4022 [00:00<?, ? examples/s]

Map:   0%|          | 0/4719 [00:00<?, ? examples/s]

# Training Arguments

In [10]:

from transformers import TrainingArguments, IntervalStrategy

training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=500,
)

In [11]:
from transformers import Trainer

trainer = Trainer(
    model=model_lora,  # Use LoRA fine-tuned model
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
)

trainer.train()

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: taran16501 (taran16501-loyalist-college) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
500,2.838900
1000,1.886100
1500,1.689400
2000,1.476100
2500,1.465300
3000,1.458600


TrainOutput(global_step=3018, training_loss=1.8007159684966778, metrics={'train_runtime': 1398.9442, 'train_samples_per_second': 8.625, 'train_steps_per_second': 2.157, 'total_flos': 3163681088077824.0, 'train_loss': 1.8007159684966778, 'epoch': 3.0})

In [12]:
results = trainer.evaluate()
print(results)

{'eval_loss': 2.5720834732055664, 'eval_runtime': 20.9139, 'eval_samples_per_second': 225.639, 'eval_steps_per_second': 28.211, 'epoch': 3.0}


#Generating Responses from a Fine-Tuned Model Using Tokenization and Inference

In [13]:
def generate_response(question):
    inputs = tokenizer(question, return_tensors="pt")
    # Move the input tensors to the same device as the model
    inputs = {k: v.to(model_lora.device) for k, v in inputs.items()}
    outputs = model_lora.generate(input_ids=inputs["input_ids"], max_length=100)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Sample response
sample_question = "What is the effect of ivabradine in heart failure after myocardial infarction?"
print(generate_response(sample_question))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


What is the effect of ivabradine in heart failure after myocardial infarction?

Vabradine is a potent inhibitor of the myocardial infarction (MI) pathway. It is a potent inhibitor of the myocardial infarction (MI) pathway. It is a potent inhibitor of the myocardial infarction (MI) pathway. It is a potent inhibitor of the myocardial infarction (MI) pathway. It is a


In [14]:
pip install evaluate bert-score rouge-score

# Evaluate the model

In [15]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import evaluate
from bert_score import score as bert_score
from rouge_score import rouge_scorer
from sklearn.metrics import roc_auc_score # Import roc_auc_score

# Load model and tokenizer
model_name = "gpt2"  # You can change this to your fine-tuned model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Ensure the model is on the correct device (GPU if available)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Set the padding token
tokenizer.pad_token = tokenizer.eos_token

# Load ROUGE and BERTScore metrics from 'evaluate' library
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

# Batch Evaluation Function for ROUGE-L and BERTScore
def evaluate_batch(questions, reference_answers, batch_size=16):
    # Batch process for faster evaluation
    rouge_scores = []
    bert_f1_scores = []
    accuracies = []
    auc_scores = []
    similarity_scores = []

    # Generate responses in batches
    generated_answers = []
    for i in tqdm(range(0, len(questions), batch_size), desc="Evaluating"):
        batch_questions = questions[i:i+batch_size]
        batch_references = reference_answers[i:i+batch_size]

        # Tokenize inputs and generate responses
        inputs = tokenizer(batch_questions, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        outputs = model.generate(input_ids=inputs["input_ids"], max_length=100)
        generated_batch = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

        generated_answers.extend(generated_batch)

        # ROUGE-L Score Calculation
        rouge_scorer_instance = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
        rouge_batch = [
            rouge_scorer_instance.score(reference, generated)['rougeL'].fmeasure
            for reference, generated in zip(batch_references, generated_batch)
        ]
        rouge_scores.extend(rouge_batch)

        # BERTScore Calculation
        P, R, F1 = bert_score(generated_batch, batch_references, lang="en")
        bert_f1_scores.extend(F1.tolist())

        # Accuracy Calculation (exact match)
        accuracy_batch = [
            1 if generated == reference else 0
            for reference, generated in zip(batch_references, generated_batch)
        ]
        accuracies.extend(accuracy_batch)

        # AUC Calculation (based on BERTScore similarity as relevance)
        # Convert BERT F1 score to relevance score (treat F1 score as a probability of relevance)
        auc_batch = [
            1 if f1_score >= 0.5 else 0
            for f1_score in F1.tolist()  # Use BERT F1 score as a proxy for relevance
        ]
        auc_scores.extend(auc_batch)
        similarity_scores.extend(F1.tolist())  # Store the similarity scores for AUC

    # Average the results
    avg_rouge_l = sum(rouge_scores) / len(rouge_scores)
    avg_bert_f1 = sum(bert_f1_scores) / len(bert_f1_scores)
    avg_accuracy = sum(accuracies) / len(accuracies)

    # Calculate AUC using the stored similarity scores and true labels
    auc = roc_auc_score(accuracies, similarity_scores)

    return {"rouge_l": avg_rouge_l, "bert_f1": avg_bert_f1}

# Sample data for evaluation (replace with your test set)
questions = ["What is the effect of ivabradine in heart failure after myocardial infarction?"] * 10  # Example question
reference_answers = ["Ivabradine improves heart rate reduction and outcomes in heart failure patients post-myocardial infarction."] * 10  # Example answers

# Evaluate the model
results = evaluate_batch(questions, reference_answers, batch_size=4)
print("Evaluation Results:", results)

Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Evaluating:  33%|███▎      | 1/3 [01:14<02:28, 74.29s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Evaluating:  67%|██████▋   | 2/3 [01:16<00:31, 31.88s/it]The attention mask and the pad token id were not set. As a consequence, you may obs

Evaluation Results: {'rouge_l': 0.15189873417721517, 'bert_f1': 0.8498231172561646, 'accuracy': 0.0, 'auc': nan}



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


#The model performed moderately well with a ROUGE-L score of 0.1519 and a good BERT F1 score of 0.8498

In [16]:
pip install lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 9.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=577041e4bb1543a93926654096c6ca00f45cf6d99ddfbcff14ea0832be31cd07
  Stored in directory: /root/.cache/pip/wheels/85/fa/a3/9c2d44c9f3cd77cf4e533b58900b2bf4487f2a17e8ec212a3d
Successfully built lime


In [ ]:
from lime.lime_text import LimeTextExplainer
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import numpy as np

# Load the model and tokenizer (You already have this in your code)
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Ensure tokenizer has a padding token (necessary for padding during inference)
tokenizer.pad_token = tokenizer.eos_token

# Define a function to generate responses for LIME
def generate_response_for_lime(question_batch):
    # Tokenize input
    inputs = tokenizer(question_batch, padding=True, truncation=True, return_tensors="pt", max_length=128)  # Reduce max_length to 128
    inputs = {key: val.to(model.device) for key, val in inputs.items()}

    # Generate model output
    with torch.no_grad():  # Avoid backpropagation during inference
        outputs = model.generate(input_ids=inputs["input_ids"], max_length=100)

    # Decode outputs to text
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

# Initialize LIME explainer
explainer = LimeTextExplainer(class_names=["relevant", "irrelevant"])

# Function to predict using the model
def predict_fn(texts):
    """
    Function to be passed into LIME. It takes a list of texts, tokenizes them, and returns the model predictions.
    """
    # We want to return a list of predictions, which should be in the format expected by LIME
    responses = generate_response_for_lime(texts)
    # For simplicity, let's treat the model's output as 'relevant' if it contains more than 10 words
    return np.array([[1.0] if len(response.split()) > 10 else [0.0] for response in responses])

# Select a batch of questions to interpret using LIME
questions_to_explain = [
    "What is the effect of ivabradine in heart failure after myocardial infarction?",
    "How does aspirin help prevent heart attacks?",
    "What are the side effects of metformin in diabetes treatment?"
]  # Example questions for LIME interpretation

# Apply LIME to explain predictions
explanation = explainer.explain_instance(
    questions_to_explain[0],  # Take the first question as an example for LIME explanation
    predict_fn,
    num_features=10,  # Limit the number of features to explain for better performance
    top_labels=1  # We only want to explain the top prediction
)

# Show explanation
explanation.show_in_notebook()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A 

LLM Prompts
1. How to implement GPT2 in the model
2. Its showing memory error how to solve it ?